# MCP setup on Mac

Run this once before `mcp_langchain.ipynb`, and again whenever you add a new MCP server.

| Server | Runs | Needs |
|---|---|---|
| Fetch | Local (stdio) | `uv` (gives `uvx`) |
| GitHub | Remote (http) | `GITHUB_PERSONAL_ACCESS_TOKEN` in `.env` |
| LangChain Docs | Remote (http) | Nothing |

| # | Step |
|---|---|
| 1 | Install `uv` (two options) |
| 2 | Check the keys in `.env` |
| 3 | Pre-download the Fetch server |
| 4 | Smoke test all three servers |
| 5 | Test the Fetch pages of the lab |
| 6 | Add a new server |

## 1. Install `uv`

Pick **one** option.

| | Option A: current environment (recommended) | Option B: Homebrew (whole Mac) |
|---|---|---|
| Command | `%pip install uv` | `brew install uv` |
| Where it lives | Inside this notebook's Python environment | `/opt/homebrew/bin`, for every project |
| Restart needed | Kernel only | Quit and reopen VS Code (Cmd+Q) |

We do **not** pip-install the Fetch server itself here: it needs the MCP SDK 2.x, while `langchain-mcp-adapters` needs 1.x. `uvx` runs each server in its own small environment, so the two never clash.

In [1]:
# Option A: uv inside this environment (recommended)
%pip install uv

# Option B: uv for the whole Mac with Homebrew (use instead of A)
# !brew install uv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import shutil

# Same lookup as in the lab notebook: Homebrew uvx first, else the one in this environment
UVX = shutil.which("uvx") or os.path.join(os.path.dirname(sys.executable), "uvx")
print("PASS" if os.path.exists(UVX) else "MISSING", "uvx ->", UVX)

PASS uvx -> /opt/anaconda3/envs/bootcamp-env/bin/uvx


## 2. Keys in `.env`

The `.env` file sits next to the notebooks and is never committed.

**Create the GitHub token** (read-only, public repos only):
1. GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** → Generate new token.
2. Name: `ironhack-mcp-lab`. Expiration: 7 days.
3. Repository access: **Public repositories (read-only)**. Add no permissions.
4. Generate, copy, and add it to `.env`:

```
OPENAI_API_KEY=sk-...
GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...
```

In [3]:
import subprocess
from dotenv import load_dotenv

load_dotenv()

# PASS / MISSING only, never the key itself
for key in ["OPENAI_API_KEY", "GITHUB_PERSONAL_ACCESS_TOKEN"]:
    print("PASS   " if os.getenv(key) else "MISSING", key)

# .env must be git-ignored (it is in this repo's .gitignore)
ignored = subprocess.run(["git", "check-ignore", "-q", ".env"]).returncode == 0
print("PASS   " if ignored else "FAIL   ", ".env is git-ignored")

PASS    OPENAI_API_KEY
PASS    GITHUB_PERSONAL_ACCESS_TOKEN
PASS    .env is git-ignored


## 3. Pre-download the Fetch server

The first `uvx` run downloads the server. Doing it here keeps the first lab call fast. It should end with the server's help text.

In [4]:
!"{UVX}" --quiet mcp-server-fetch@2026.10.24 --help

usage: mcp-server-fetch [-h] [--user-agent USER_AGENT] [--ignore-robots-txt]
                        [--proxy-url PROXY_URL] [--allow-private-ips]

give a model the ability to make web requests

options:
  -h, --help            show this help message and exit
  --user-agent USER_AGENT
                        Custom User-Agent string
  --ignore-robots-txt   Ignore robots.txt restrictions
  --proxy-url PROXY_URL
                        Proxy URL to use for requests
  --allow-private-ips   Allow fetching private, loopback and link-local
                        addresses (refused by default)


## 4. Smoke test all three servers

Each server is tested **on its own**, so one failure does not hide the others. PASS means: the server started (or answered), the MCP handshake worked and it listed its tools.

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

SERVERS = {
    "fetch": {
        "transport": "stdio",
        "command": UVX,
        "args": ["mcp-server-fetch@2026.10.24"],
    },
    "github": {
        "transport": "http",
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {os.getenv('GITHUB_PERSONAL_ACCESS_TOKEN')}",
            "X-MCP-Toolsets": "repos,issues",
            "X-MCP-Readonly": "true",
        },
    },
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp",
    },
}

# Unwrap grouped async errors to the first real cause
def root_cause(error):
    while getattr(error, "exceptions", None):
        error = error.exceptions[0]
    return f"{type(error).__name__}: {error}"

for name, config in SERVERS.items():
    try:
        tools = await MultiServerMCPClient({name: config}).get_tools()
        print(f"PASS {name}: {len(tools)} tools -> {[t.name for t in tools][:5]}")
    except Exception as error:
        print(f"FAIL {name}: {root_cause(error)}")

PASS fetch: 1 tools -> ['fetch']
PASS github: 19 tools -> ['get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_release_by_tag']
PASS langchain-docs: 3 tools -> ['search_docs_by_lang_chain', 'query_docs_filesystem_docs_by_lang_chain', 'submit_feedback']


## 5. Test the Fetch pages of the lab

Fetch respects each site's robots.txt, so a site can refuse it. This calls the `fetch` tool directly (no LLM, free) on the two pages the lab uses. If one FAILs, use the project's README on GitHub instead, e.g. `https://raw.githubusercontent.com/openai/openai-agents-python/main/README.md`.

In [6]:
PAGES = [
    "https://docs.langchain.com/oss/python/langgraph/overview",
    "https://openai.github.io/openai-agents-python/",
]

fetch_tool = (await MultiServerMCPClient({"fetch": SERVERS["fetch"]}).get_tools())[0]

for url in PAGES:
    result = await fetch_tool.ainvoke({"url": url, "max_length": 300})
    text = result[0]["text"]
    print("FAIL" if text.startswith("Error") else "PASS", url)
    print("    ", text[:200].replace("\n", " "))

PASS https://docs.langchain.com/oss/python/langgraph/overview
     Contents of https://docs.langchain.com/oss/python/langgraph/overview: <p>LangGraph overview - Docs by LangChain</p>  > ## Documentation Index > > Fetch the complete documentation index at: /llms.txt >
PASS https://openai.github.io/openai-agents-python/
     Contents of https://openai.github.io/openai-agents-python/: <p>OpenAI Agents SDK</p>  Skip to content  # OpenAI Agents SDK  The OpenAI Agents SDK enables you to build agentic AI apps in a lightweight,


## 6. Add a new MCP server

1. **Find it:** the company's own docs first, then [registry.modelcontextprotocol.io](https://registry.modelcontextprotocol.io), then [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers).
2. **Check it:** who made it, is it maintained, what can it reach, where does the data go.
3. **Read its README** for the connection:
   - a URL → remote: `"transport": "http"` + `"url"` (token in `"headers"`, token itself in `.env`)
   - `uvx ...` → local Python: `"command": UVX` + `"args"`
   - `npx ...` → local JavaScript: `"command": "npx"` + `"args"` (needs Node.js: `brew install node`)
4. **Pin the version** (cell below).
5. **Add it to `SERVERS`** in step 4, run the smoke test, then copy the same config into `mcp_langchain.ipynb`.

Examples (copy into `SERVERS`):

```python
"time": {"transport": "stdio", "command": UVX, "args": ["mcp-server-time@<version>"]},
"filesystem": {"transport": "stdio", "command": "npx",
               "args": ["-y", "@modelcontextprotocol/server-filesystem@<version>", "<folder path>"]},
```

In [7]:
# Latest version to pin (change the name): Python servers on PyPI
!pip index versions mcp-server-time 2>/dev/null | head -1

mcp-server-time (2026.10.19)


## Troubleshooting

| Symptom | Fix |
|---|---|
| `MISSING uvx` after Option A | Restart the kernel and re-run step 1 |
| `MISSING uvx` after Option B | Quit VS Code fully (Cmd+Q) and reopen it |
| `FAIL github: ... 401` or `400` | Token missing, wrong or expired: redo step 2, restart the kernel |
| `FAIL fetch: TimeoutError` or a very slow first call | Run step 3 first |
| A Fetch page FAILs with robots.txt | The site blocks automated fetching: use the README link from step 5 |